# LLM Fundamentals

In this notebook, we will learn:

1. What Large Language Models (LLMs) do
2. Groq API Integration
3. Message Roles
4. Streaming Responses
5. Token Counting
6. Temperature
7. LLM Strategy and Model Selection

By the end of this notebook you should understand how modern chat models work and how to choose the right model for different workloads.

# What is an LLM?

A Large Language Model (LLM) is a machine learning model trained on massive amounts of text.

Input:
    Text / Messages

Output:
    Generated Text

Examples:

User:
"What is AWS?"

LLM:
"AWS is Amazon Web Services..."

Modern LLMs operate using a sequence of messages instead of simple text prompts.

Examples:
- ChatGPT
- Claude
- Gemini
- Llama
- DeepSeek
- Groq-hosted Models

LangChain represents conversations using messages that contain:

- Role
- Content
- Metadata

This message-based design makes it easier to manage conversations and agent workflows.

In [1]:
# Install required packages

%pip install -q langchain langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Configure Environment Variables

Create a .env file:

GROQ_API_KEY=your_api_key_here

Never hardcode API keys inside notebooks.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Key Loaded:", bool(GROQ_API_KEY))

API Key Loaded: True


# Groq API

Groq provides extremely fast inference for open-source models.

Popular Groq-hosted models:

- Llama 3
- Llama 4
- DeepSeek
- Qwen

Benefits:

- Low latency
- OpenAI-compatible APIs
- Excellent for prototyping

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CB4BCBFCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CB4BE5CAD0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

# Message Roles

Messages are the fundamental unit of context in LangChain.

Common roles:

1. System
2. User (Human)
3. Assistant (AI)

System:
    Controls behavior

User:
    Provides instructions

Assistant:
    Model response

Conversation = Sequence of Messages

In [4]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

messages = [
    SystemMessage(
        content="You are an AWS solutions architect."
    ),
    HumanMessage(
        content="Explain Amazon Bedrock."
    )
]

response = llm.invoke(messages)

print(response.content)

Amazon Bedrock is a fully managed service offered by AWS that allows developers to build and deploy AI-powered applications using pre-trained foundation models. These models are trained on large datasets and can be fine-tuned for specific use cases, enabling developers to create a wide range of AI-powered applications, such as chatbots, text analysis tools, and more.

With Amazon Bedrock, developers can access a library of pre-trained foundation models, including language models, computer vision models, and other types of AI models. These models can be used to perform tasks such as text classification, sentiment analysis, object detection, and more.

The key benefits of Amazon Bedrock include:

1. **Simplified AI development**: Amazon Bedrock provides pre-trained models that can be easily integrated into applications, reducing the need for extensive AI expertise and development time.
2. **Cost-effective**: By using pre-trained models, developers can avoid the high costs associated with

# Understanding Message Flow

System Message
    ↓

Human Message
    ↓

Model Processing
    ↓

AI Message

The system message generally appears first and establishes the assistant's behavior.

In [5]:
messages = [
    SystemMessage(
        content="Answer in one sentence."
    ),
    HumanMessage(
        content="What is Retrieval Augmented Generation?"
    )
]

response = llm.invoke(messages)

print(response.content)

Retrieval Augmented Generation (RAG) is a type of natural language processing model that combines the strengths of retrieval-based and generation-based approaches to produce more accurate and informative text outputs by retrieving relevant information from a knowledge base and then generating text based on that information.


# Temperature

Temperature controls randomness.

Range:
0.0 → Deterministic
2.0 → Highly Creative

Typical Usage:

0.0 - 0.3
    Factual Q&A
    Coding
    SQL Generation

0.4 - 0.8
    General Chatbots

0.8 - 1.5
    Creative Writing

Higher temperature increases variation.
Lower temperature improves consistency.

In [6]:
creative_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.5
)

creative_response = creative_model.invoke(
    "Write a futuristic startup idea."
)

print(creative_response.content)

**Startup Name: DreamWeaver Neurotech**

**Idea:** DreamWeaver Neurotech is a futuristic startup that develops non-invasive, AI-powered brain-computer interfaces (BCIs) to monitor, analyze, and influence human dreams. By harnessing the power of artificial intelligence and neural networking, our technology enables users to control, manipulate, and even share their dreams.

**Key Features:**

1. **Neural Headband:** A wearable, non-invasive headband with EEG sensors and AI-powered neuroprocessors that track brain activity during sleep, identifying and categorizing different stages of dreaming.
2. **Dream Analysis:** Our proprietary algorithm analyzes the brain activity data to decode and interpret the contents of the user's dreams, providing insights into their subconscious mind.
3. **Dream Engineering:** Users can choose from a library of pre-designed dream scenarios or create their own using our intuitive dream-building interface. The AI system can then inject these scenarios into the 

# Streaming

Normally:

Request
    ↓
Wait
    ↓
Complete Response

Streaming:

Request
    ↓
Token
Token
Token
Token
    ↓
Complete Response

Streaming improves user experience because users see output immediately instead of waiting for the full response.

In [7]:
for chunk in llm.stream(
    "Explain vector databases in simple terms."
):
    print(chunk.content, end="", flush=True)

**What is a Vector Database?**

A vector database is a type of database that stores and manages complex data, such as images, videos, and text, in a unique way. Instead of storing the actual data, it stores a mathematical representation of the data, called a vector.

**Think of it like a map:**

Imagine you have a map with many locations marked on it. Each location is represented by a set of coordinates (latitude and longitude). You can use these coordinates to find the location on the map, even if you don't have the actual map.

Similarly, a vector database stores the "coordinates" of your data in a high-dimensional space. These coordinates, or vectors, allow the database to quickly find and compare similar data points, even if they are complex and high-dimensional.

**Key Features:**

1. **Vector-based search**: Vector databases enable fast and efficient search and retrieval of similar data points.
2. **High-dimensional data**: They can handle complex data with many features or dimen

# Why Streaming Matters

Advantages:

- Faster perceived response time
- Better UX
- Supports long outputs
- Useful for chat applications
- Useful for agent workflows

Most production chat systems use streaming.

# Token Counting

LLMs process text as tokens.

Example:

"Hello World"

may become

["Hello", "World"]

Pricing is usually:

Input Tokens +
Output Tokens

Understanding token usage is critical for:
- Cost optimization
- Context management
- Performance monitoring

In [8]:
response = llm.invoke(
    "Explain Amazon S3 in two sentences."
)

print(response.content)

print("\nUsage Metadata:")
print(response.usage_metadata)

Amazon S3 (Simple Storage Service) is a cloud-based object storage service provided by Amazon Web Services (AWS) that allows users to store and retrieve large amounts of data in the form of objects, such as files, images, and videos. With S3, users can store, manage, and analyze their data in a scalable, durable, and highly available manner, making it a popular choice for a wide range of applications, from data lakes and analytics to website hosting and backup storage.

Usage Metadata:
{'input_tokens': 44, 'output_tokens': 99, 'total_tokens': 143}


# Example Cost Optimization

Bad Candidate:
    Complex model for every request

Better Strategy:

Simple Questions
    ↓
Cheap Model

Complex Reasoning
    ↓
Powerful Model

Benefits:

- Reduced cost
- Better latency
- Scalability

# LLM Strategy

Not every problem requires the largest model.

Choose based on:

1. Complexity
2. Cost
3. Latency
4. Accuracy Requirements

Examples:

Simple:
- Summarization
- Classification
- Extraction

Complex:
- Multi-step reasoning
- Agent workflows
- Research assistants

A routing layer can decide which model should be used.

In [9]:
def choose_model(task_complexity):
    
    if task_complexity == "simple":
        return "llama-3.1-8b"

    elif task_complexity == "medium":
        return "llama-3.3-70b"

    else:
        return "deepseek-r1"


for task in ["simple", "medium", "complex"]:
    print(task, "->", choose_model(task))

simple -> llama-3.1-8b
medium -> llama-3.3-70b
complex -> deepseek-r1


# Dynamic Model Selection Architecture

User Request
      ↓

Complexity Evaluation
      ↓

Routing Layer
      ↓

Simple Task --------> Small Model

Complex Task -------> Large Model

Reasoning Task -----> Reasoning Model

This pattern significantly reduces cost while maintaining quality.

In [10]:
def classify_request(prompt):

    prompt = prompt.lower()

    if "summarize" in prompt:
        return "simple"

    if "analyze" in prompt:
        return "complex"

    return "medium"


user_prompt = "Analyze the pros and cons of Bedrock Agents"

task_type = classify_request(user_prompt)

selected_model = choose_model(task_type)

print("Task Type:", task_type)
print("Selected Model:", selected_model)

Task Type: complex
Selected Model: deepseek-r1


# Key Takeaways

LLMs:
    Convert messages into generated text.

Messages:
    System, User, Assistant.

Groq:
    Fast inference platform for open-source models.

Temperature:
    Controls creativity.

Streaming:
    Provides token-by-token responses.

Token Counting:
    Helps monitor costs.

LLM Strategy:
    Use the smallest model capable of solving the problem.

Dynamic Model Selection:
    Route requests to different models based on complexity.

# References

Messages
https://docs.langchain.com/oss/python/langchain/messages

Streaming
https://docs.langchain.com/oss/python/langchain/streaming

Log LLM Calls
https://docs.langchain.com/langsmith/log-llm-trace

Model Providers
https://docs.langchain.com/langsmith/playground-model-providers

Dynamic Model Selection
https://docs.langchain.com/oss/python/langchain/middleware/custom#dynamic-model-selection